In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_table = "retailnova.bronze.products"
silver_table = "retailnova.silver.products"

In [0]:
df = spark.table(bronze_table)

In [0]:
# Get last processed timestamp for Silver
last_processed = spark.sql("""
    SELECT last_processed_at
    FROM retailnova.bronze.etl_control
    WHERE source_name = 'products_silver'
""").first()[0]

print("Last Silver processed:", last_processed)

Last Silver processed: 2026-08-29 00:00:00


In [0]:
# Keep only new/changed product records
new_products = df.filter(
    col("updated_at") > last_processed
)

print("New records for Silver:", new_products.count())

New records for Silver: 600


In [0]:
# Remove records without product_id
silver_df = new_products.filter(
    col("product_id").isNotNull()
)

In [0]:
silver_df = silver_df.withColumn(
    "product_name",
    trim(col("product_name"))
)

silver_df = silver_df.withColumn(
    "category",
    trim(col("category"))
)

silver_df = silver_df.withColumn(
    "category",
    when(
        lower(col("category")) == "home appliance",
        "Home Appliances"
    ).otherwise(col("category"))
)

silver_df = silver_df.withColumn(
    "subcategory",
    trim(col("subcategory"))
)

silver_df = silver_df.withColumn(
    "brand",
    trim(col("brand"))
)

silver_df = silver_df.withColumn(
    "supplier_id",
    trim(col("supplier_id"))
)



In [0]:
silver_df = silver_df.withColumn(
    "unit_price",
    col("unit_price").cast("double")
)

silver_df = silver_df.withColumn(
    "cost_price",
    col("cost_price").cast("double")
)

In [0]:
silver_df = silver_df.withColumn(
    "quality_status",
    when(
        col("unit_price").isNull() |
        col("cost_price").isNull() |
        (col("unit_price") < 0) |
        (col("cost_price") < 0),
        "INVALID"
    ).otherwise("VALID")
)

In [0]:
# First run creates the table
if not spark.catalog.tableExists(silver_table):

    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(silver_table)

else:

    silver_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(silver_table)

In [0]:
# Get the actual latest timestamp in Silver
new_watermark = spark.sql(f"""
    SELECT MAX(updated_at)
    FROM {silver_table}
""").first()[0]

print("New Silver watermark:", new_watermark)

New Silver watermark: 2026-09-11


In [0]:
spark.sql(f"""
    UPDATE retailnova.bronze.etl_control
    SET last_processed_at = TIMESTAMP('{new_watermark}')
    WHERE source_name = 'products_silver'
""")

print("Silver products updated successfully")

Silver products updated successfully


In [0]:
print("Silver product count:")

spark.sql("""
    SELECT COUNT(*)
    FROM retailnova.silver.products
""").show()

Silver product count:
+--------+
|COUNT(*)|
+--------+
|   10600|
+--------+



In [0]:
%sql
SELECT *
FROM retailnova.bronze.etl_control;

source_name,last_processed_at
customers_silver,2026-09-10T23:59:10.000Z
customers,2026-09-10T23:59:10.000Z
products_silver,2026-09-11T00:00:00.000Z
products,2026-09-11T00:00:00.000Z


In [0]:
%sql
SELECT * FROM retailnova.silver.products

product_id,product_name,category,subcategory,brand,unit_price,cost_price,supplier_id,updated_at,quality_status
P000001,Product 000001,Beauty,Skincare,Brand_024,1757.03,1251.23,S00359,2026-07-09,VALID
P000002,Product 000002,Home Appliances,Cooling,Brand_198,1651.11,828.65,S00291,2026-08-07,VALID
P000003,Product 000003,Toys,Learning,Brand_163,686.52,393.82,S00453,2026-08-20,VALID
P000004,Product 000004,Beauty,Personal Care,Brand_121,4891.03,2562.2,S00296,2026-07-16,VALID
P000005,Product 000005,Fashion,Men,Brand_029,497.36,308.28,S00339,2026-07-17,VALID
P000006,Product 000006,Home Appliances,Cleaning,Brand_092,366.79,189.55,S00268,2026-08-29,VALID
P000007,Product 000007,Beauty,Skincare,Brand_120,1316.06,1001.12,S00093,2026-07-07,VALID
P000008,Product 000008,Electronics,null,Brand_101,199.11,137.24,S00218,2026-07-27,VALID
P000009,Product 000009,Fashion,Men,Brand_174,1540.74,795.87,S00379,2026-08-19,VALID
P000010,Product 000010,Home Appliances,Small Appliances,Brand_188,937.11,489.27,S00445,2026-08-13,VALID
